<a href="https://colab.research.google.com/github/SebastianoDenegri/Exploratory_Data_Analysis_for_wine_dataset/blob/main/A_Systemic_Risk_Framework_for_the_Vietnamese_Market_a_Composite_Risk_Index_and_Financial_Stress_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import the needed libraries.

In [1]:
import requests
import pandas as pd
from functools import reduce
from io import StringIO
import time
import yfinance as yf
try:
  import akshare as ak
except:
  !pip install akshare
  import akshare as ak

Build a function to collect Vietnamese stock data from <a href='https://cafef.vn/'>CafeF</a>.

In [2]:
# Helper function to get historical data from cafef
start="01/01/2012"
end="31/03/2026"
def fetch_price_history(ticker, start_date = start, end_date = end):
    url = (
        f"https://s.cafef.vn/Ajax/PageNew/DataHistory/PriceHistory.ashx"
        f"?Symbol={ticker}&StartDate={start_date}&EndDate={end_date}"
        f"&PageIndex=1&PageSize=10000"
    )
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    data = r.json().get("Data", {}).get("Data", [])

    if not data:
        return pd.DataFrame(columns=["Date", f"{ticker}_close", f"{ticker}_volume"])

    df = pd.DataFrame(data).rename(columns={
        "Ngay": "Date",
        "GiaDieuChinh": f"{ticker}_close",
        "KhoiLuongKhopLenh": f"{ticker}_volume"
    })

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df[f"{ticker}_adj_closing_price"] = pd.to_numeric(df[f"{ticker}_close"], errors="coerce")
    df[f"{ticker}_trading_volume"] = pd.to_numeric(df[f"{ticker}_volume"], errors="coerce")

    end_dt = pd.to_datetime(end_date, dayfirst=True)
    df = df[df["Date"] <= end_dt]

    #return df[["Date", f"{ticker}_close", f"{ticker}_volume"]].sort_values("Date")
    return df[["Date", f"{ticker}_adj_closing_price", f"{ticker}_trading_volume"]] \
       .rename(columns={
           f"{ticker}_adj_closing_price": f"{ticker}_close",
           f"{ticker}_trading_volume": f"{ticker}_volume"
       })

Build a Data Information Table for the Vietnamese equity data, including:
- Ticker;
- Company Name;
- Sector: the industry the company operates in;
- Risk Transmission Channel: a macro-economic sector that represents a specific source or amplififier of shocks, spreading systemic risk within and across sectors.

In [3]:
ticker_info = {
    "NT2": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Energy", "Company": "PetroVietnam Power Nhon Trach 2 JSC"},
    "PVD": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Energy", "Company": "PetroVietnam Drilling & Well Services Corp"},
    "PVS": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Energy", "Company": "PetroVietnam Technical Services Corp"},
    "PVT": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Energy", "Company": "PetroVietnam Transportation Corp"},
    "REE": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Utilities / Infrastructure", "Company": "Refrigeration Electrical Engineering Corp"},
    "ACB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Asia Commercial Joint Stock Bank"},
    "CTG": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "VietinBank"},
    "MBB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Military Commercial JSB"},
    "SHB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Saigon-Hanoi Commercial JSB"},
    "STB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Saigon Thuong Tin Commercial JSB"},
    "VCB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Vietcombank"},
    "BVH": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Insurance", "Company": "Bao Viet Holdings"},
    "PVI": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Insurance", "Company": "PVI Holdings"},
    "SSI": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Securities/Brokerage", "Company": "SSI Securities Corp"},
    "VIX": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Securities/Brokerage", "Company": "VIX Securities JSC"},
    "VND": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Securities/Brokerage", "Company": "VNDirect Securities Corp"},
    "GMD": {"Risk Transmission Channel": "Logistics", "Sector": "Logistics", "Company": "Gemadept Corp"},
    "VSC": {"Risk Transmission Channel": "Logistics", "Sector": "Logistics", "Company": "Vietnam Container Shipping JSC"},
    "MSN": {"Risk Transmission Channel": "Real Economy", "Sector": "Consumer Goods / Retail", "Company": "Masan Group Corp"},
    "PNJ": {"Risk Transmission Channel": "Real Economy", "Sector": "Consumer Goods / Retail", "Company": "Phu Nhuan Jewelry JSC"},
    "VNM": {"Risk Transmission Channel": "Real Economy", "Sector": "Consumer Goods / Retail", "Company": "Vietnam Dairy Products JSC"},
    "CSM": {"Risk Transmission Channel": "Real Economy", "Sector": "Industrial / Materials", "Company": "Casumina"},
    "DPM": {"Risk Transmission Channel": "Real Economy", "Sector": "Industrial / Materials", "Company": "PetroVietnam Fertilizer and Chemicals Corp"},
    "HPG": {"Risk Transmission Channel": "Real Economy", "Sector": "Materials / Industrial", "Company": "Hoa Phat Group JSC"},
    "HSG": {"Risk Transmission Channel": "Real Economy", "Sector": "Materials / Industrial", "Company": "Hoa Sen Group"},
    "NKG": {"Risk Transmission Channel": "Real Economy", "Sector": "Materials / Industrial", "Company": "Nam Kim Steel JSC"},
    "FPT": {"Risk Transmission Channel": "Real Economy", "Sector": "Technology", "Company": "FPT Corp"},
    "DIG": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Development Investment Construction JSC"},
    "DXG": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Dat Xanh Group JSC"},
    "KBC": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Kinh Bac City Development Holding Corp"},
    "KDH": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Khang Dien House Trading and Investment JSC"},
    "VIC": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Vingroup JSC"},
}
ticker_df = (
    pd.DataFrame.from_dict(ticker_info, orient="index")
      .reset_index()
      .rename(columns={"index": "ticker"})
      [["Risk Transmission Channel", "Sector", "Company", "ticker"]]
)

ticker_df

,Risk Transmission Channel,Sector,Company,ticker
0,Energy & Infrastructure,Energy,PetroVietnam Power Nhon Trach 2 JSC,NT2
1,Energy & Infrastructure,Energy,PetroVietnam Drilling & Well Services Corp,PVD
2,Energy & Infrastructure,Energy,PetroVietnam Technical Services Corp,PVS
3,Energy & Infrastructure,Energy,PetroVietnam Transportation Corp,PVT
4,Energy & Infrastructure,Utilities / Infrastructure,Refrigeration Electrical Engineering Corp,REE
5,Financial System,Finance - Banking,Asia Commercial Joint Stock Bank,ACB
6,Financial System,Finance - Banking,VietinBank,CTG
7,Financial System,Finance - Banking,Military Commercial JSB,MBB
8,Financial System,Finance - Banking,Saigon-Hanoi Commercial JSB,SHB
9,Financial System,Finance - Banking,Saigon Thuong Tin Commercial JSB,STB


Collect the Vietnamese equity data.

In [4]:
# fetch vn tickers
dfs = [fetch_price_history(t) for t in ticker_info.keys()]

# merge all into one dataframe
df_vn = reduce(lambda left, right: pd.merge(left, right, on="Date", how="outer"), dfs)
df_vn = df_vn.sort_values("Date").reset_index(drop=True)
df_vn.head(5)

,Date,NT2_close,NT2_volume,PVD_close,PVD_volume,PVS_close,PVS_volume,PVT_close,PVT_volume,REE_close,...,DIG_close,DIG_volume,DXG_close,DXG_volume,KBC_close,KBC_volume,KDH_close,KDH_volume,VIC_close,VIC_volume
0,2012-01-03,1.06,16500.0,13.42,40680.0,3.73,77900.0,1.01,33120.0,2.77,...,2.04,105040.0,1.17,30690.0,4.96,315520.0,3.60,55300.0,7.81,39190.0
1,2012-01-04,1.06,12000.0,12.79,97220.0,3.50,285700.0,1.04,457630.0,2.85,...,2.00,142770.0,1.15,28520.0,5.01,70540.0,3.73,61980.0,7.85,38760.0
2,2012-01-05,0.97,0.0,12.74,66780.0,3.42,446500.0,1.07,621160.0,2.85,...,1.96,66330.0,1.13,33310.0,4.91,43340.0,3.90,52010.0,7.49,15860.0
3,2012-01-06,1.06,2900.0,12.74,15050.0,3.34,94100.0,1.04,282320.0,2.80,...,1.91,242390.0,1.12,13890.0,4.91,29410.0,3.86,37940.0,7.45,22080.0
4,2012-01-09,1.10,1100.0,12.58,31050.0,3.29,125400.0,1.04,231330.0,2.90,...,1.85,630070.0,1.10,6770.0,4.82,59880.0,3.81,53980.0,7.81,14940.0


In [5]:
df_vn.tail(5)

,Date,NT2_close,NT2_volume,PVD_close,PVD_volume,PVS_close,PVS_volume,PVT_close,PVT_volume,REE_close,...,DIG_close,DIG_volume,DXG_close,DXG_volume,KBC_close,KBC_volume,KDH_close,KDH_volume,VIC_close,VIC_volume
3547,2026-03-25,27.70,2979600.0,34.75,3336600.0,42.2,7282100.0,21.70,8810800.0,70.9,...,13.80,11696900.0,14.15,20550300.0,28.9,4274000.0,25.85,5073300.0,128.7,3013400.0
3548,2026-03-26,27.75,1455700.0,34.70,4219300.0,42.0,5057500.0,22.40,13267800.0,71.7,...,13.55,8624500.0,13.85,15497700.0,28.7,1780000.0,25.45,2692100.0,130.0,2435600.0
3549,2026-03-27,27.90,950600.0,35.75,5186400.0,42.8,5938400.0,22.30,9317400.0,71.7,...,14.45,24713800.0,14.60,41747300.0,30.7,6091600.0,26.40,4704200.0,132.6,2124700.0
3550,2026-03-30,27.90,1371700.0,36.20,4041300.0,42.8,5262900.0,21.85,12201400.0,70.0,...,14.20,13413500.0,14.30,16998400.0,31.8,7130700.0,26.05,4370900.0,129.5,2042500.0
3551,2026-03-31,27.60,846800.0,34.80,5738900.0,40.9,7568200.0,21.80,7238400.0,68.5,...,14.25,11416400.0,14.55,21018600.0,31.9,2270100.0,26.00,4193600.0,135.0,3316700.0


In [6]:
df_vn.shape

(3552, 65)

Gather VNIndex historical data (we'll use it to construct a dummy "Financial Sress Proxy* variable).

In [7]:
df_vnindex = fetch_price_history("VNINDEX")
df_vnindex = df_vnindex.sort_values("Date").reset_index(drop=True)
df_vnindex.head(10)

,Date,VNINDEX_close,VNINDEX_volume
0,2012-01-03,350.00,13737030
1,2012-01-04,348.84,19643360
2,2012-01-05,340.94,17221570
3,2012-01-06,336.73,19779270
4,2012-01-09,339.32,19198950
5,2012-01-10,344.68,24725890
6,2012-01-11,347.43,20793880
7,2012-01-12,348.11,18662540
8,2012-01-13,354.33,16398250
9,2012-01-16,357.87,20302930


In [8]:
df_vnindex.tail()

,Date,VNINDEX_close,VNINDEX_volume
3543,2026-03-25,1658.19,862805200
3544,2026-03-26,1644.63,701366700
3545,2026-03-27,1672.80,861666600
3546,2026-03-30,1662.54,793800500
3547,2026-03-31,1674.49,857297400


In [9]:
df_vnindex.shape

(3548, 3)

In [10]:
foreign_index = {
    "^GSPC": "S&P 500 (US)",
    "^GSPTSE": "S&P/TSX (Canada)",
    "^STOXX50E": "EURO STOXX 50 (EU)",
    "^FTSE": "FTSE 100 (UK)",
    # "000300.SS": "CSI 300 (China)",
    "^N225": "Nikkei 225 (Japan)",
    "^KS11": "KOSPI (S.Korea)",
    "^AXJO": "S&P/ASX 200 (Australia)",
    "^HSI": "Hang Seng (HK)",
    "^TWII": "TWSE (Taiwan)"
}
foreign_index.keys()
df_fx = yf.download(list(foreign_index.keys()), start="2012-01-01", end="2026-04-01", auto_adjust= True)[['Close']]#, 'Volume']]

df_fx.columns =  [f"{ticker}_{level}" for level, ticker in df_fx.columns]

[*********************100%***********************]  9 of 9 completed


In [11]:
df_fx

,^AXJO_Close,^FTSE_Close,^GSPC_Close,^GSPTSE_Close,^HSI_Close,^KS11_Close,^N225_Close,^STOXX50E_Close,^TWII_Close
Date,,,,,,,,,
2012-01-02,NaN,NaN,NaN,NaN,NaN,1826.369995,NaN,NaN,NaN
2012-01-03,4101.200195,5699.899902,1277.060059,12208.400391,18877.410156,1875.410034,NaN,2389.909912,7053.348145
2012-01-04,4187.799805,5668.500000,1277.300049,12226.500000,18727.310547,1866.219971,8560.110352,2349.889893,7082.937988
2012-01-05,4142.700195,5624.299805,1281.060059,12237.400391,18813.410156,1863.739990,8488.709961,2315.750000,7130.827148
2012-01-06,4108.500000,5649.700195,1277.810059,12188.599609,18593.060547,1843.140015,8390.349609,2298.649902,7120.477051
...,...,...,...,...,...,...,...,...,...
2026-03-25,8557.599609,10106.799805,6591.899902,32382.599609,25335.949219,5642.209961,53749.621094,5649.330078,33439.109375
2026-03-26,8525.700195,9972.200195,6477.160156,31887.500000,24856.429688,5460.459961,53603.648438,5565.930176,33337.621094
2026-03-27,8516.299805,9967.400391,6368.850098,31960.699219,24951.880859,5438.870117,53373.070312,5505.799805,33112.589844


In [12]:
df_CSI300= (
    ak.stock_zh_index_daily(symbol="sh000300")
      .assign(date=lambda x: pd.to_datetime(x["date"]))
      .set_index("date")
      .loc["2012-01-01":"2026-03-31", ["close"]]#, "volume"]]
      .rename_axis(None)
      .rename(columns={"close": "CSI300_close"})#, "volume": "CSI300_volume"})
      .reset_index()
      .rename(columns={"index": "Date", "date": "Date"})
      .set_index("Date")
)

In [13]:
df_CSI300

,CSI300_close
Date,
2012-01-04,2298.753
2012-01-05,2276.385
2012-01-06,2290.601
2012-01-09,2368.570
2012-01-10,2447.349
...,...
2026-03-25,4537.466
2026-03-26,4477.534
2026-03-27,4502.570


In [14]:
# combine all international data
df_glob = (df_vnindex.merge(df_fx, on="Date", how="outer")
.merge(df_CSI300, on="Date", how="outer")
.sort_values("Date")
.reset_index(drop=True)).drop(['VNINDEX_volume'], axis=1)

df_glob = df_glob.set_index("Date").ffill().reset_index()
df_glob.tail()

,Date,VNINDEX_close,^AXJO_Close,^FTSE_Close,^GSPC_Close,^GSPTSE_Close,^HSI_Close,^KS11_Close,^N225_Close,^STOXX50E_Close,^TWII_Close,CSI300_close
3701,2026-03-25,1658.19,8557.599609,10106.799805,6591.899902,32382.599609,25335.949219,5642.209961,53749.621094,5649.330078,33439.109375,4537.466
3702,2026-03-26,1644.63,8525.700195,9972.200195,6477.160156,31887.500000,24856.429688,5460.459961,53603.648438,5565.930176,33337.621094,4477.534
3703,2026-03-27,1672.80,8516.299805,9967.400391,6368.850098,31960.699219,24951.880859,5438.870117,53373.070312,5505.799805,33112.589844,4502.570
3704,2026-03-30,1662.54,8461.000000,10128.000000,6343.720215,31934.900391,24750.789062,5277.299805,51885.851562,5541.790039,32518.160156,4491.950
3705,2026-03-31,1674.49,8481.799805,10176.500000,6528.520020,32768.000000,24788.140625,5052.459961,51063.718750,5569.729980,31722.990234,4450.049


In [15]:
df_glob.head()

,Date,VNINDEX_close,^AXJO_Close,^FTSE_Close,^GSPC_Close,^GSPTSE_Close,^HSI_Close,^KS11_Close,^N225_Close,^STOXX50E_Close,^TWII_Close,CSI300_close
0,2012-01-02,NaN,NaN,NaN,NaN,NaN,NaN,1826.369995,NaN,NaN,NaN,NaN
1,2012-01-03,350.00,4101.200195,5699.899902,1277.060059,12208.400391,18877.410156,1875.410034,NaN,2389.909912,7053.348145,NaN
2,2012-01-04,348.84,4187.799805,5668.500000,1277.300049,12226.500000,18727.310547,1866.219971,8560.110352,2349.889893,7082.937988,2298.753
3,2012-01-05,340.94,4142.700195,5624.299805,1281.060059,12237.400391,18813.410156,1863.739990,8488.709961,2315.750000,7130.827148,2276.385
4,2012-01-06,336.73,4108.500000,5649.700195,1277.810059,12188.599609,18593.060547,1843.140015,8390.349609,2298.649902,7120.477051,2290.601


In [16]:
df_glob.shape

(3706, 12)